In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))

In [ ]:
from ib_insync import *
from ibkr.Class_IBKR_IB import IBKR_IB
ibkr = IBKR_IB(port=7496)

async def start_ibkr():
    await ibkr.connect()
    print("IBKR connected:", ibkr.ib.isConnected())

await start_ibkr()

Error 1100, reqId -1: Connectivity between IBKR and Trader Workstation has been lost.
Error 1100, reqId -1: Connectivity between IBKR and Trader Workstation has been lost.
Peer closed connection.


In [ ]:
stock_symbols = ['IBIT']

futures_symbols = ['BTCM6',
                   'BTCN6',
                   'BTCQ6',
                   'BTCU6',
                   'BTCV6',
                   'BTCX6']



In [ ]:
futures_contracts = []

for symbol in futures_symbols:
    contract = Future(localSymbol=symbol, exchange='CME', currency="USD")
    await ibkr.ib.qualifyContractsAsync(contract) # this may lead to a printed line since its return has nowhere to be mapped
    futures_contracts.append(contract)
    print(contract)

In [ ]:
stock_contracts = []
stock_details = []
option_chains = []

for symbol in stock_symbols:
    contract = Stock(symbol, 'SMART', "USD")
    await ibkr.ib.qualifyContractsAsync(contract) # this may lead to a printed line since its return has nowhere to be mapped
    stock_contracts.append(contract)
    print(contract)

    details = await ibkr.ib.reqContractDetailsAsync(contract)
    stock_details.append(details)
    print(details)

    option_chain = await ibkr.ib.reqSecDefOptParamsAsync(underlyingSymbol=details[0].contract.symbol,
                                                         futFopExchange="",
                                                         underlyingSecType=details[0].contract.secType,
                                                         underlyingConId=details[0].contract.conId
                                                         )
    option_chains.append(option_chain)
    print(option_chain)
    print(option_chain[0].expirations)
    print(option_chain[0].strikes)

    print('\n')

In [ ]:
option_contracts = []

for chain, details in zip(option_chains, stock_details):
    for expiry in chain[0].expirations:
        for strike in chain[0].strikes:
            for right in ['C', 'P']:
                option_contract = Option(lastTradeDateOrContractMonth=expiry,
                                         strike=float(strike),
                                         right=right,
                                         symbol=details[0].contract.symbol,
                                         exchange=details[0].contract.exchange,
                                         currency=details[0].contract.currency,
                                        )
                option_contracts.append(option_contract)

await ibkr.ib.qualifyContractsAsync(*option_contracts) # this may lead to printed lines since its return has nowhere to be mapped


In [12]:
rows = []

all_contracts = [*futures_contracts, *stock_contracts]#, *option_contracts] 

for contract in all_contracts:
    ticker = ibkr.ib.reqMktData(contract, snapshot=True)#, genericTickList="100,101,588")
    row = {
            "conId": getattr(ticker.contract, "conId", None),
            "secType": getattr(ticker.contract, "secType", None),
            "symbol": getattr(ticker.contract, "symbol", None),
            "expiration": getattr(ticker.contract, "lastTradeDateOrContractMonth", None),
            "strike": getattr(ticker.contract, "strike", None),
            "right": getattr(ticker.contract, "right", None),

            "close": getattr(ticker, "close", None),
        #    "volume": getattr(ticker, "volume", None),
        #    "avgDailyVolume": getattr(ticker, "xyz", None)
            "futuresOpenInterest": getattr(ticker, "futuresOpenInterest", None),
            "putOpenInterest": getattr(ticker, "putOpenInterest", None),
            "callOpenInterest": getattr(ticker, "callOpenInterest", None),
        }

    rows.append(row)

import asyncio
await asyncio.sleep(15)

import pandas as pd
df_all = pd.DataFrame(rows)
df_all

,conId,secType,symbol,expiration,strike,right,close,futuresOpenInterest,putOpenInterest,callOpenInterest
0,751356962,FUT,BRR,20260626,0.0,,62945.000000,NaN,NaN,NaN
1,850790355,FUT,BRR,20260731,0.0,,63240.000000,NaN,NaN,NaN
2,859040542,FUT,BRR,20260828,0.0,,63495.000000,NaN,NaN,NaN
3,772435574,FUT,BRR,20260925,0.0,,63745.000000,NaN,NaN,NaN
4,876880607,FUT,BRR,20261030,0.0,,64115.000000,NaN,NaN,NaN
5,887699043,FUT,BRR,20261127,0.0,,64425.000000,NaN,NaN,NaN
6,677037673,STK,IBIT,,0.0,,36.360001,NaN,NaN,NaN
